In [ ]:
!git clone https://github.com/fashn-AI/fashn-vton-1.5.git
%cd fashn-vton-1.5

In [ ]:
!pip install -e .
!pip install onnxruntime-gpu==1.26.0
!pip install gradio

In [ ]:
!python scripts/download_weights.py --weights-dir ./weights

In [ ]:
import sys
sys.path.append("src")
import torch
import gradio as gr
from PIL import Image
from fashn_vton import TryOnPipeline

# 1. Initialize Pipeline
pipeline = TryOnPipeline(weights_dir="./weights")

# 2. Force float16 (FP16) execution for 2-3x speedup on T4 GPU
pipeline.inference_dtype = torch.float16
pipeline.tryon_model.to(dtype=torch.float16)

# 3. Compile tryon model using torch 2.0 to optimize kernel operations
pipeline.tryon_model = torch.compile(pipeline.tryon_model, mode="reduce-overhead")

# 4. Warm up compilation so subsequent generations are immediate
print("Warming up and compiling try-on model...")
try:
    dummy_img = Image.new("RGB", (768, 1024), (255, 255, 255))
    pipeline(
        person_image=dummy_img,
        garment_image=dummy_img,
        category="tops",
        num_timesteps=10,
        seed=42
    )
    print("Model compilation and warm-up successful!")
except Exception as e:
    print(f"Warm-up skipped: {e}")

def predict(person_image, garment_image, category, garment_photo_type, num_samples, num_timesteps, guidance_scale, seed, segmentation_free):
    if person_image is None or garment_image is None:
        return None
    result = pipeline(
        person_image=person_image.convert("RGB"),
        garment_image=garment_image.convert("RGB"),
        category=category,
        garment_photo_type=garment_photo_type,
        num_samples=num_samples,
        num_timesteps=num_timesteps,
        guidance_scale=guidance_scale,
        seed=int(seed),
        segmentation_free=segmentation_free,
    )
    return result.images

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# FASHN VTON v1.5 Virtual Try-On (Optimized for Colab T4)")
    with gr.Row():
        with gr.Column():
            person_input = gr.Image(type="pil", label="Person Image")
            garment_input = gr.Image(type="pil", label="Garment Image")
            category_input = gr.Dropdown(choices=["tops", "bottoms", "one-pieces"], value="tops", label="Category")
            garment_type_input = gr.Dropdown(choices=["model", "flat-lay"], value="model", label="Garment Photo Type")
            with gr.Accordion("Advanced Settings", open=False):
                samples_input = gr.Slider(minimum=1, maximum=4, step=1, value=1, label="Number of Samples")
                steps_input = gr.Slider(minimum=10, maximum=50, step=1, value=20, label="Number of Timesteps")
                guidance_input = gr.Slider(minimum=1.0, maximum=5.0, step=0.1, value=1.5, label="Guidance Scale")
                seed_input = gr.Number(value=42, label="Seed")
                seg_input = gr.Checkbox(value=True, label="Segmentation Free")
            btn = gr.Button("Generate Try-On")
        with gr.Column():
            gallery_output = gr.Gallery(label="Generated Images")

    btn.click(
        fn=predict,
        inputs=[
            person_input,
            garment_input,
            category_input,
            garment_type_input,
            samples_input,
            steps_input,
            guidance_input,
            seed_input,
            seg_input
        ],
        outputs=gallery_output
    )

demo.launch(share=True)
